# Looking at relationship between October 1 soil moisture and JAS precipitation and temperature

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import FancyArrowPatch
from matplotlib.widgets import Slider, RadioButtons
from glob import glob

### grab butte snotel data

In [4]:
east_snotels = glob("/scratch/dlhogan/ess-project-data/meteorological-data/SNOTEL/*_sntl_*.csv")

In [142]:
east_df = pd.DataFrame()
for f in east_snotels:
    site_name = f.split("/")[-1].split("_")[0]
    df = pd.read_csv(f)
    df["site"] = site_name
    east_df = pd.concat([east_df, df], ignore_index=True)

east_df["datetime"] = pd.to_datetime(east_df["datetime"])
east_df.set_index(["datetime", "site"], inplace=True)

# drop site 737
east_df = east_df[~east_df.index.get_level_values("site").isin(["737"])]

# add water year
east_df["water_year"] = east_df.index.get_level_values("datetime").year.where(east_df.index.get_level_values("datetime").month < 10, 
                                                                              east_df.index.get_level_values("datetime").year + 1)

east_df = east_df[east_df.water_year >=2003]

In [201]:
# resample to yearly sum and mean 
jas_precipitation = east_df[east_df.index.get_level_values("datetime").month.isin([7,8,9])][["PRECIPITATION","water_year"]].groupby("water_year").sum()
jas_temperature = east_df[east_df.index.get_level_values("datetime").month.isin([9])][["AIR TEMP","water_year"]].groupby("water_year").mean()
oct_1_soil_moisture_2in = east_df[(east_df.index.get_level_values("datetime").month == 9) & 
        (east_df.index.get_level_values("datetime").day == 30)][["SOIL MOISTURE -2IN","water_year"]].groupby("water_year").last()['SOIL MOISTURE -2IN']
oct_1_soil_moisture_8in = east_df[(east_df.index.get_level_values("datetime").month == 9) & 
        (east_df.index.get_level_values("datetime").day == 30)][["SOIL MOISTURE -8IN","water_year"]].groupby("water_year").last()['SOIL MOISTURE -8IN']
oct_1_soil_moisture_20in = east_df[(east_df.index.get_level_values("datetime").month == 9) & 
        (east_df.index.get_level_values("datetime").day == 30)][["SOIL MOISTURE -20IN","water_year"]].groupby("water_year").last()['SOIL MOISTURE -20IN']

# rename columns
oct_1_soil_moisture_2in.name = "soil_moisture"
oct_1_soil_moisture_8in.name = "soil_moisture"
oct_1_soil_moisture_20in.name = "soil_moisture"

# calculate z-score 
jas_precipitation_z = ((jas_precipitation - jas_precipitation.mean()) / jas_precipitation.std())['PRECIPITATION']
jas_precipitation_z.name = "precip"
jas_temperature_z = ((jas_temperature - jas_temperature.mean()) / jas_temperature.std())['AIR TEMP']
jas_temperature_z.name = "temp"
oct_1_sm_z = ((oct_1_soil_moisture_8in - oct_1_soil_moisture_8in.mean()) / oct_1_soil_moisture_8in.std())
oct_1_sm_z.name = "soil_moisture"


In [202]:
df = pd.concat([jas_precipitation_z, jas_temperature_z, oct_1_sm_z], axis=1).dropna()

In [203]:
from nb_tools import soil_moisture_quadrant

In [204]:
soil_moisture_quadrant.plot_soil_moisture_quadrant(df.reset_index())